## Final evaluation

The 2023 validation set was used for tuning.  
After choosing the ANN and Random Forest configurations, I train them again on 2015-2023 and evaluate them once on 2024-2025.

The test set is not used to choose hyperparameters.

In [2]:
from pathlib import Path 

import numpy as np 
import pandas as pd 
import tensorflow as tf 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, 
    balanced_accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    log_loss, 
    confusion_matrix, 
    classification_report
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation 
from tensorflow.keras import Input 
from tensorflow.keras import optimizers

In [4]:
np.random.seed(42)
tf.random.set_seed(42)

In [5]:
PROCESSED_DATA_DIR = Path("../data/processed")

input_file = (PROCESSED_DATA_DIR / "feature_split_matches.csv")

matches = pd.read_csv(input_file, parse_dates=["Date"])

matches = matches.sort_values(
    by=["Date", "MatchID"]
).reset_index(drop=True)

print("Dataset shape:", matches.shape)
print(matches["DataSplit"].value_counts())

Dataset shape: (26536, 49)
DataSplit
Training      18810
Test           5119
Validation     2607
Name: count, dtype: int64


In [6]:
final_train_data = matches.loc[
    matches["DataSplit"].isin(
        ["Training", "Validation"]
    )
].copy()

test_data = matches.loc[
    matches["DataSplit"] == "Test"
].copy()

In [8]:
print(
    "Final training matches:", 
    len(final_train_data)
)

print("Test matches:", len(test_data))

print(
    "Final training period: ",
    final_train_data["Date"].min(), 
    "-", 
    final_train_data["Date"].max()
)

print(
    "Test period: ", 
    test_data["Date"].min(), 
    "-", 
    test_data["Date"].max()
)

Final training matches: 21417
Test matches: 5119
Final training period:  2015-01-05 00:00:00 - 2023-12-31 00:00:00
Test period:  2024-01-01 00:00:00 - 2025-11-16 00:00:00


In [9]:
assert (
    final_train_data["Date"].max() < test_data["Date"].min()
)

print("Final chronological split is correct.")

Final chronological split is correct.


In [10]:
numeric_features = [
    "Player1Rank",
    "Player2Rank",
    "Player1Points",
    "Player2Points",

    "Player1MatchesBefore",
    "Player2MatchesBefore",
    "ExperienceDifference",

    "Player1WinRateBefore",
    "Player2WinRateBefore",
    "WinRateDifference",

    "Player1Recent5WinRate",
    "Player2Recent5WinRate",
    "Recent5WinRateDifference",

    "Player1SurfaceMatchesBefore",
    "Player2SurfaceMatchesBefore",

    "Player1SurfaceWinRateBefore",
    "Player2SurfaceWinRateBefore",
    "SurfaceWinRateDifference",

    "H2HMatchesBefore",
    "Player1H2HWinRateBefore",
    "Player2H2HWinRateBefore",
    "DifferenceH2HWinRateBefore",

    "Player1DaysSinceLastMatch",
    "Player2DaysSinceLastMatch",

    "RankDifference",
    "PointsDifference"
]

In [11]:
X_final_train = final_train_data[numeric_features].copy()
y_final_train = final_train_data["Player1Won"].to_numpy()

X_test = test_data[numeric_features].copy()
y_test= test_data["Player1Won"].to_numpy()

In [12]:
print("Final training X:", X_final_train.shape)
print("Final training y:", y_final_train.shape)
print("Test X:", X_test.shape)
print("Test y:", y_test.shape)


Final training X: (21417, 26)
Final training y: (21417,)
Test X: (5119, 26)
Test y: (5119,)


In [13]:
final_training_medians = (X_final_train.median())
X_final_train = (
    X_final_train.fillna(
        final_training_medians
    )
)

X_test = X_test.fillna(final_training_medians)

In [15]:
print(
    "Missing final training values:", 
    X_final_train.isna().sum().sum()
)

print(
    "Missing test values:", 
    X_test.isna().sum().sum()
)

Missing final training values: 0
Missing test values: 0


### Final ANN

The best validation configuration from `ann_classifier.ipynb` is:

- hidden layers: 32 and 16 neurons
- activation: ReLU
- learning rate: 0.0005
- epochs: 100

I create a new network from scratch and train it on the combined training + validation period.

In [ ]:
normalizer = StandardScaler()
ANN_train_X = normalizer.fit_transform(X_final_train)
ANN_test_X = normalizer.transform(X_test)

In [ ]:
num_instances, num_features = (ANN_train_X.shape)
print("Training matches:", num_instances)
print("Number of features:", num_features)

In [ ]:
# initialize final network
final_ann = Sequential()

# input
final_ann.add(Input(shape=(num_features,)))

# hidden layers
final_ann.add(Dense(32))
final_ann.add(Activation("relu"))

final_ann.add(Dense(16))
final_ann.add(Activation("relu"))

# output
final_ann.add(Dense(1))
final_ann.add(Activation("sigmoid"))

In [ ]:
final_ann.compile(
    optimizer=optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
final_ann_history = final_ann.fit(
    x=ANN_train_X,
    y=y_final_train,
    epochs=100,
    verbose=1
)

In [ ]:
# use the test data 
ann_test_probabilities = (
    final_ann.predict(
        ANN_test_X, 
        verbose=0
    ).reshape(-1)
)

ann_test_predictions = (
    ann_test_probabilities >= 0.5
).astype(int)

In [ ]:
ann_test_results = {
    "Model": "ANN", 
    "Accuracy": accuracy_score(
        y_test, ann_test_predictions
    ), 
    "BalancedAccuracy": balanced_accuracy_score(
        y_test, ann_test_predictions
    ), 
    "Precision": precision_score(
        y_test, ann_test_predictions, zero_division=0
    ), 
    "Recall": recall_score(
        y_test, ann_test_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_test, ann_test_predictions, zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_test, ann_test_probabilities
    ), 
    "LogLoss": log_loss(
        y_test, ann_test_probabilities
    )
}

In [ ]:
pd.DataFrame([ann_test_results])

## final random forest
selected configuration: 
200 trees
maximum depth = 10
minimum samples per leaf = 1
max_features = sqrt

In [ ]:
final_rf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    min_samples_leaf=1, 
    max_features="sqrt", 
    random_state=42, 
    n_jobs=-1
)

In [ ]:
final_rf.fit(X_final_train, y_final_train)

rf_test_predictions = (final_rf.predict(X_test))
rf_test_probabilities = (final_rf.predict_proba(X_test)[:, 1])


In [ ]:
rf_test_results = {
    "Model": "Random Forest", 
    "Accuracy": accuracy_score(
        y_test, rf_test_predictions
    ),
    "BalancedAccuracy": balanced_accuracy_score(
        y_test, rf_test_predictions
    ),
    "Precision": precision_score(
        y_test, rf_test_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, rf_test_predictions, zero_division=0
    ), 
    "F1": f1_score(
        y_test, rf_test_predictions, zero_division=0
    ), 
    "ROC_AUC": roc_auc_score(
        y_test, rf_test_probabilities
    ), 
    "LogLoss": log_loss(
        y_test, rf_test_probabilities
    )
}

pd.DataFrame([rf_test_results])

In [ ]:
## final comparison 

final_model_results = pd.DataFrame(
    [
        ann_test_results, 
        rf_test_results
    ]
)

final_model_results = (
    final_model_results.sort_values(
        by="Accuracy", ascending= False
    ).reset_index(drop=True)
)

final_model_results

In [ ]:
ranking_test_predictions = np.where(
    test_data["RankDifference"] > 0, 
    1, 
    0
)

equal_rank = (
    test_data["RankDifference"] == 0
)

ranking_test_predictions[equal_rank] = np.where(
    test_data.loc[
        equal_rank, "PointsDifference"
    ] >= 0, 
    1, 
    0
)

In [ ]:
ranking_test_results = {
    "Model": "Better-ranked player", 
    "Accuracy": accuracy_score(
        y_test, ranking_test_predictions
    ), 
    "BalancedAccuracy": balanced_accuracy_score(
        y_test, ranking_test_predictions
    ),
    "Precision": precision_score(
        y_test, ranking_test_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, ranking_test_predictions, zero_division=0
    ), 
    "F1": f1_score(
        y_test, ranking_test_predictions, zero_division=0
    )
}

In [ ]:
test_comparison = pd.DataFrame(
    [
        {
            key: ann_test_results[key]
            for key in [
                "Model",
                "Accuracy",
                "BalancedAccuracy",
                "Precision",
                "Recall",
                "F1"
            ]
        }, 
        {
            key: rf_test_results[key]
            for key in [
                "Model",
                "Accuracy",
                "BalancedAccuracy",
                "Precision",
                "Recall",
                "F1"
            ]
        }, 
        ranking_test_results
    ]
)

test_comparison = (
    test_comparison.sort_values(
        by="Accuracy", ascending=False
    ).reset_index(drop=True)
)

test_comparison

In [ ]:
print("ANN")
print(
    classification_report(
        y_test, 
        ann_test_predictions, 
        target_names=[
            "Player 2 wins", 
            "Player 1 wins"
        ], 
        zero_division=0
    )
)

In [ ]:
print("Random Forest")
print(
    classification_report(
        y_test, 
        rf_test_predictions, 
        target_names=[
            "Player 2 wins", 
            "Player 1 wins"
        ], 
        zero_division=0
    )
)

In [ ]:
ann_matrix = confusion_matrix(y_test, ann_test_predictions)

fig, axes = plt.subplots(
    figsize =(6, 6),
    tight_layout=True
)

sns.heatmap(
    ann_matrix, 
    square=True, 
    annot=True, 
    fmt="d",
    cbar=False, 
    cmap="Blues"
)

axes.set_xlabel("Predicted Label")
axes.set_ylabel("True Label")
axes.set_title("ANN Test Confusion Matrix")
plt.show()

In [ ]:
rf_matrix = confusion_matrix(y_test, rf_test_predictions)

fig, axes = plt.subplots(
    figsize=(6, 6), 
    tight_layout=True
)

sns.heatmap(
    rf_matrix, 
    square=True, 
    annot=True, 
    fmt="d",
    cbar=False, 
    cmap="Blues"
)

axes.set_xlabel("Predicted Label")
axes.set_ylabel("True Label")
axes.set_title("Random Forest Test Confusion Matrix")
plt.show()

In [ ]:
final_feature_importance = pd.DataFrame(
    {
        "Feature": numeric_features, 
        "Importance": final_rf.feature_importances_
    }
)

final_feature_importance = (
    final_feature_importance.sort_values(
        by="Importance", ascending=False
    ).reset_index(drop=True)
)

final_feature_importance

In [ ]:
top_features = (
    final_feature_importance.head(15).sort_values(
        by="Importance", ascending=True
    )
)

plt.figure(figsize=(9, 7))

plt.barh(
    top_features["Feature"], 
    top_features["Importance"]
)

plt.title("Final Random Forest Feature Importance")
plt.xlabel("Importance")

plt.tight_layout()
plt.show()

In [ ]:
test_analysis = test_data[
    [
        "MatchID",
        "Date",
        "Tournament",
        "Surface",
        "Round",
        "Player1",
        "Player2",

        "Player1Rank",
        "Player2Rank",
        "RankDifference",

        "PointsDifference",

        "WinRateDifference",
        "Recent5WinRateDifference",
        "SurfaceWinRateDifference",

        "H2HMatchesBefore",

        "Player1Won"
    ]
].copy()

In [ ]:
##add ANN results 
test_analysis["ANNProbability"] = ann_test_probabilities
test_analysis["ANNPrediction"] = ann_test_predictions
test_analysis["ANNCorrect"] = (
    test_analysis["ANNPrediction"] == test_analysis["Player1Won"]
)
test_analysis["ANNConfidence"] = np.maximum(
    test_analysis["ANNProbability"], 
    1- test_analysis["ANNProbability"]
)

In [ ]:
#add RF results
test_analysis["RFProbability"] = rf_test_probabilities
test_analysis["RFPrediction"] = rf_test_predictions
test_analysis["RFCorrect"] = (
    test_analysis["RFPrediction"] == test_analysis["Player1Won"]
)
test_analysis["RFConfidence"] = np.maximum(
    test_analysis["RFProbability"], 
    1 - test_analysis["RFProbability"]
)


In [ ]:
ann_most_wrong = (
    test_analysis.loc[
        test_analysis["ANNCorrect"] == False
    ].sort_values(
        by="ANNConfidence", ascending=False
    )
)

ann_most_wrong[
    [
        "Date",
        "Tournament",
        "Surface",
        "Player1",
        "Player2",
        "Player1Rank",
        "Player2Rank",
        "Player1Won",
        "ANNPrediction",
        "ANNProbability",
        "ANNConfidence"
    ]
].head(10)

In [ ]:
ann_most_correct = (
    test_analysis.loc[
        test_analysis["ANNCorrect"] == True
    ].sort_values(
        by="ANNConfidence", ascending=False
    )
)

ann_most_correct[
    [
        "Date",
        "Player1",
        "Player2",
        "Player1Won",
        "ANNProbability",
        "ANNConfidence"
    ]
].head(15)

In [ ]:
rf_most_wrong = (
    test_analysis.loc[
        test_analysis["RFCorrect"] == False
    ].sort_values(
        by="RFConfidence", ascending=False
    )
)

rf_most_wrong[
    [
        "Date",
        "Tournament",
        "Surface",
        "Player1",
        "Player2",
        "Player1Rank",
        "Player2Rank",
        "Player1Won",
        "RFPrediction",
        "RFProbability",
        "RFConfidence"
    ]
].head(10)

In [ ]:
rf_most_correct = (
    test_analysis.loc[
        test_analysis["RFCorrect"] == True
    ].sort_values(
        by="RFConfidence", ascending=False
    )
)

rf_most_correct[
    [
        "Date",
        "Player1",
        "Player2",
        "Player1Won",
        "RFProbability",
        "RFConfidence"
    ]
].head(10)

### Most confident correct and wrong predictions

The confident errors are interesting because several of them have a very large ranking gap.  
In thosee matches the model strongly follows the usual indicators of player strength, but the lower-ranked player wins anyway.

So the biggest mistakes often look like **upsets**, while the most confident correct predictions usually involve a clear favorite. This is consistent for both the ANN and the Random Forest.

In [ ]:
test_analysis["AbsoluteRankDifference"] = (
    test_analysis["RankDifference"].abs()
)

test_analysis["AbsolutePointsDifference"] = (
    test_analysis["PointsDifference"].abs()
)

test_analysis["AbsoluteWinRateDifference"] = (
    test_analysis["WinRateDifference"].abs()
)

test_analysis["AbsoluteSurfaceDifference"] = (
    test_analysis["SurfaceWinRateDifference"].abs()
)

In [ ]:
features_to_compare= [
    "AbsoluteRankDifference", 
    "AbsolutePointsDifference", 
    "AbsoluteWinRateDifference", 
    "AbsoluteSurfaceDifference", 
    "H2HMatchesBefore"
]

rf_correct_wrong_comparisons = (
    test_analysis.groupby(
        "RFCorrect"
    )[features_to_compare].mean().T
)

rf_correct_wrong_comparisons 

The Random Forest is more often correct when the two players are clearly different.

For example, correct predictions have larger average differences in ranking, ranking points, previous win rate and surface win rate.  
So close matches are harder for the model.

`H2HMatchesBefore` changes very little between correct and wrong predictions, so head-to-head history looks less useful here.

Means can be influenced by a few extreme values, so I also compare the medians.

In [ ]:
rf_correct_wrong_median = (
    test_analysis.groupby(
        "RFCorrect"
    )[features_to_compare].median().T
)

rf_correct_wrong_median

The medians show the same general pattern:

- ranking difference: 35 for wrong predictions vs 44 for correct ones
- points difference: 515 vs 920
- previous win-rate difference: 0.0808 vs 0.1118
- surface win-rate difference: 0.0914 vs 0.1199
- median head-to-head matches: 0 in both groups

This supports the idea that the models work better when one player has a clearer advantage.  
The head-to-head feature is less informative because many matches have no previous direct meeting.

In [ ]:
ann_correct_wrong_comparisons = (
    test_analysis.groupby(
        "ANNCorrect"
    )[features_to_compare].mean().T
)

ann_correct_wrong_comparisons

In [ ]:
ann_wrong_correct_median = (
    test_analysis.groupby(
        "ANNCorrect"
    )[features_to_compare].median().T
)

ann_wrong_correct_median

The ANN gives almost the same pattern as the Random Forest: correct predictions have larger player-strength differences, while closer matches produce more errors.  
This suggests that the difficulty comes mainly from uncertain matches rather than from one specific model.

In [ ]:
RESULTS_DIR = Path("../results")
MODELS_DIR = Path("../models")

final_model_results.to_csv(
    RESULTS_DIR / "final_model_results.csv", index=False
)

test_comparison.to_csv(
    RESULTS_DIR / "final_test_comparison.csv", index=False
)

final_feature_importance.to_csv(
    RESULTS_DIR / "final_feature_importance.csv", index=False
)

test_analysis.to_csv(
    RESULTS_DIR / "final_test_analysis.csv", index=False
)


In [ ]:
import joblib 

joblib.dump(final_rf, MODELS_DIR / "final_RF.joblib")
final_ann.save(MODELS_DIR / "final_ann.keras")
joblib.dump(normalizer, MODELS_DIR / "final_ann_scaler.joblib")

In [ ]:
print("FINAL TEST RESULTS")
print(final_model_results)

In [ ]:
print("INCLUDING RANKING BASELINE")
print(test_comparison)

In [ ]:
print("TOP 15 FEATURES")
print(final_feature_importance.head(15))

In [ ]:
print("RF CORRECT VS WRONG")
print(rf_correct_wrong_comparisons)

print("\nANN CORRECT VS WRONG")
print(ann_correct_wrong_comparisons)